# 03 Why collection fails: testing the explanations

Question: which of five explanations for failed household waste collection in Lagos fits the available evidence?

| ID | Explanation | Evidence used here |
|----|-------------|--------------------|
| H1 | Resident non-payment | Survey: payment status against time since last collection [S17] |
| H2 | Too few operators | Residents per PSP operator by LGA [S18, S19, S20] against news failure reports and survey gaps |
| H3 | Poor road access | Survey: road condition and truck access against time since last collection [S17]; news reports of road access problems |
| H4 | Dumpsite bottleneck | Straight-line distance from each LGA to the nearest active disposal site [S21, A09] against news failure reports and survey gaps; news reports of dumpsite access problems |
| H5 | Inequality | Operator coverage across LGAs; survey home type and gated estate against service [S17] |

Two outcome measures are available, and both are weak. News failure reports (`complaints.csv`) reflect where outlets reported, not where failure is most common. The survey is a pilot convenience sample of 28 households. LGA-level tests use 20 LGAs and Spearman rank correlations; survey results are reported as counts. No result here establishes cause.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import pandas as pd

from lagos_waste import charts, geo
from lagos_waste import hypotheses as hy
from lagos_waste import news_analysis as na
from lagos_waste.paths import FIGURES, MAPS
from lagos_waste.sites import SITES_PATH

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
SOURCE_COVERAGE = "Source: LAWMA PSP directory, January 2024 [S18]; NPC 2006 census [S19]. Boundaries: GRID3 via geoBoundaries [S16]."
SOURCE_SITES = "Source: OpenStreetMap, retrieved 4 October 2026 [S21]; site status [S22, S24]. Straight-line distance [A09]."
SOURCE_SURVEY = "Source: resident survey, pilot convenience sample, n = 28, 3 to 4 October 2026 [S17]."


## LGA indicator table

In [2]:
ind = hy.build_indicators()
ind.to_csv(hy.INDICATORS_PATH, index=False)
ind[["lga", "psp_operators", "psp_ward_slots", "population_2006_npc", "residents_per_operator_npc",
     "residents_per_operator_state", "km_to_nearest_site", "nearest_site", "articles", "failure_articles",
     "survey_responses", "survey_gap_over_14_days"]].sort_values("residents_per_operator_npc", ascending=False)

,lga,psp_operators,psp_ward_slots,population_2006_npc,residents_per_operator_npc,residents_per_operator_state,km_to_nearest_site,nearest_site,articles,failure_articles,survey_responses,survey_gap_over_14_days
13,Lagos Island,2,4,212700,106350.0,429924.0,16.1,Olusosun,5,1,2,1
6,Epe,4,15,181734,45434.0,80908.0,52.4,Ewu-Elepe,2,0,1,1
1,Ajeromi-Ifelodun,17,18,687316,40430.0,84429.0,15.0,Solous,10,8,1,0
5,Badagry,6,19,237731,39622.0,63403.0,34.2,Solous,5,1,1,1
16,Ojo,16,18,609173,38073.0,58845.0,14.7,Solous,2,0,0,0
15,Mushin,19,21,631857,33256.0,69554.0,7.0,Olusosun,11,4,2,1
0,Agege,16,17,461743,28859.0,64566.0,7.4,Olusosun,7,4,2,0
12,Kosofe,25,28,682772,27311.0,37385.0,1.9,Olusosun,5,3,0,0
18,Shomolu,15,17,403569,26905.0,68342.0,5.6,Olusosun,3,2,0,0
14,Lagos Mainland,13,17,326700,25131.0,48421.0,10.6,Olusosun,6,1,0,0


## H2: too few operators

The LAWMA directory lists 431 distinct operators across 56 LGA and LCDA areas [S18]. Lagos Island LGA appears only as "Lagos Island East", so its operator count is incomplete (C09) and it is excluded from the correlations. Correlations are repeated with the Lagos State population series because the two 2006 counts disagree by LGA (C08).

In [3]:
h2 = ind[ind["lga"] != "Lagos Island"]
hy.spearman_table(h2, ["residents_per_operator_npc", "residents_per_operator_state", "residents_per_slot_npc"],
                  ["failure_articles", "failure_articles_per_million_npc"])

,x,y,n,rho,p_value
0,residents_per_operator_npc,failure_articles,19,-0.17,0.481
1,residents_per_operator_npc,failure_articles_per_million_npc,19,-0.41,0.079
2,residents_per_operator_state,failure_articles,19,-0.07,0.781
3,residents_per_operator_state,failure_articles_per_million_npc,19,-0.26,0.289
4,residents_per_slot_npc,failure_articles,19,0.22,0.374
5,residents_per_slot_npc,failure_articles_per_million_npc,19,-0.12,0.631


In [4]:
shapes = geo.load_lagos_lgas()
mapped = geo.join_counts(shapes, ind.assign(residents_per_operator_thousands=(ind["residents_per_operator_npc"] / 1000).round(1)),
                         "residents_per_operator_thousands")
mapped.loc[mapped["lga"] == "Lagos Island", "residents_per_operator_thousands"] = float("nan")  # incomplete (C09)
geo.choropleth_png(mapped, "residents_per_operator_thousands",
                   "Residents per PSP operator by LGA (thousands), 2006 population, January 2024 operators",
                   "Residents per operator (thousands)", SOURCE_COVERAGE + " Lagos Island not shown: directory incomplete (C09).",
                   FIGURES / "03_map_residents_per_operator.png")
geo.choropleth_html(mapped, "residents_per_operator_thousands", "Residents per PSP operator (thousands)",
                    SOURCE_COVERAGE, MAPS / "03_residents_per_operator.html");

In [5]:
spread = ind[ind["lga"] != "Lagos Island"].set_index("lga")["residents_per_operator_npc"]
print(f"Lowest: {spread.idxmin()} {spread.min():,.0f}; highest: {spread.idxmax()} {spread.max():,.0f}; "
      f"ratio {spread.max() / spread.min():.1f}")

Lowest: Eti-Osa 3,835; highest: Epe 45,434; ratio 11.8


## H4: dumpsite bottleneck

Active disposal sites in 2026: Olusosun (Ikeja) and Solous (Igando, Alimosho), both described as at the end of their operating lives [S24], and Ewu-Elepe (Ikorodu) [S22]. Epe landfill was closed for conversion to a waste-to-energy plant [S23]. Solous and Ewu-Elepe are matched to unnamed OpenStreetMap landfill polygons [A11].

In [6]:
sites = pd.read_csv(SITES_PATH)
sites[["site", "lga", "status_2026", "lat", "lon", "match"]]

,site,lga,status_2026,lat,lon,match
0,Olusosun,Ikeja,"operating, end of life",6.59450,3.37693,named in OpenStreetMap
1,Solous,Alimosho,"operating, end of life",6.56421,3.25271,unnamed polygon nearest Igando [A11]
2,Ewu-Elepe,Ikorodu,operating,6.59938,3.58029,unnamed polygon nearest Elepe [A11]


In [7]:
hy.spearman_table(ind, ["km_to_nearest_site"], ["failure_articles", "articles", "failure_articles_per_million_npc"])

,x,y,n,rho,p_value
0,km_to_nearest_site,failure_articles,20,-0.62,0.004
1,km_to_nearest_site,articles,20,-0.53,0.017
2,km_to_nearest_site,failure_articles_per_million_npc,20,-0.25,0.297


In [8]:
dist_map = geo.join_counts(shapes, ind, "km_to_nearest_site")
geo.choropleth_png(dist_map, "km_to_nearest_site",
                   "Straight-line distance from each LGA to the nearest active disposal site, 2026",
                   "Kilometres", SOURCE_SITES, FIGURES / "03_map_site_distance.png", points=sites);

In [9]:
survey = pd.read_csv(hy.SURVEY_PATH, dtype=str, keep_default_na=False).merge(
    ind[["lga", "km_to_nearest_site"]], on="lga", how="left")
known = survey[survey["gap_over_14_days"] != ""].copy()
known["distance_band"] = pd.cut(known["km_to_nearest_site"], [0, 10, 100], labels=["10 km or less", "over 10 km"])
band = pd.crosstab(known["distance_band"], known["gap_over_14_days"])
print(band)
print(known.groupby("gap_over_14_days")["km_to_nearest_site"].agg(["count", "mean", "median"]).round(1))
charts.grouped_hbar(
    ["LGA 10 km or less from a site", "LGA over 10 km from a site"],
    [("Last collection over 14 days ago", list(band["yes"])), ("Last collection 14 days ago or less", list(band["no"]))],
    "Time since last collection by distance from the respondent's LGA to the nearest active site",
    "Households", SOURCE_SURVEY + " " + SOURCE_SITES, FIGURES / "03_survey_gap_by_site_distance.png",
);

gap_over_14_days  no  yes
distance_band            
10 km or less      7    5
over 10 km         4    6
                  count  mean  median
gap_over_14_days                     
no                   11   6.6     7.3
yes                  11  16.7    10.7


## H1, H3 and H5 in the survey

These repeat the survey tables in notebook 02 in compact form, so that all five explanations can be compared.

In [10]:
def share(frame: pd.DataFrame, mask: pd.Series) -> str:
    # "k of n" for respondents who remembered their last collection
    sub = frame[mask & (frame["gap_over_14_days"] != "")]
    return f"{(sub['gap_over_14_days'] == 'yes').sum()} of {len(sub)}"

truck = survey["truck_user"] == "yes"
rows = [
    ("H1", "Regular payers served by the truck", share(survey, truck & (survey["pays_bill"] == "regularly"))),
    ("H1", "Non-payers and occasional payers", share(survey, survey["pays_bill"].isin(["no", "sometimes"]))),
    ("H3", "Untarred road", share(survey, survey["road_condition"] == "untarred")),
    ("H3", "Tarred road in good condition", share(survey, survey["road_condition"] == "tarred_good")),
    ("H3", "Truck cannot reach gate", share(survey, survey["truck_access"] == "no")),
    ("H3", "Truck can reach gate", share(survey, survey["truck_access"] == "yes")),
    ("H5", "Room in shared house", share(survey, survey["home_type"] == "shared_room")),
    ("H5", "Flat with 2 or more bedrooms or duplex", share(survey, survey["home_type"].isin(["flat_2plus", "duplex_detached"]))),
    ("H5", "Gated estate", share(survey, survey["gated_estate"] == "yes")),
    ("H5", "Not in gated estate", share(survey, survey["gated_estate"] == "no")),
]
pd.DataFrame(rows, columns=["hypothesis", "group", "last collection over 14 days ago"])

,hypothesis,group,last collection over 14 days ago
0,H1,Regular payers served by the truck,6 of 13
1,H1,Non-payers and occasional payers,3 of 5
2,H3,Untarred road,6 of 8
3,H3,Tarred road in good condition,3 of 10
4,H3,Truck cannot reach gate,2 of 2
5,H3,Truck can reach gate,8 of 19
6,H5,Room in shared house,2 of 3
7,H5,Flat with 2 or more bedrooms or duplex,5 of 11
8,H5,Gated estate,9 of 15
9,H5,Not in gated estate,2 of 7


In [11]:
arts = na.articles(na.load_complaints())
fail = arts[arts["is_service_failure"] == "yes"]
for p in ["road_access", "dumpsite_access", "operator_capacity", "fees_or_billing"]:
    n = fail["problem_types"].str.split(";").map(lambda v, p=p: p in v).sum()
    print(f"{p}: {n} of {len(fail)} service-failure articles")

road_access: 4 of 21 service-failure articles
dumpsite_access: 7 of 21 service-failure articles
operator_capacity: 10 of 21 service-failure articles
fees_or_billing: 10 of 21 service-failure articles


## Summary of evidence

| ID | Explanation | News record | Survey (n = 28) | Assessment |
|----|-------------|-------------|-----------------|------------|
| H1 | Resident non-payment | Fees and billing problems appear in 10 of 21 failure articles; officials most often blame residents | 6 of 13 regular payers served by the truck had waited more than 14 days, against 3 of 5 non-payers and occasional payers; no non-payer cited cost | Not supported as the main explanation: paying households also go unserved |
| H2 | Too few operators | No significant rank correlation between residents per operator and failure reports (rho between -0.41 and 0.22 across six versions, 19 LGAs, all p above 0.05) | Too few responses per LGA | Not supported at LGA level; operator counts do not measure trucks or capacity |
| H3 | Poor road access | Road access in 4 of 21 failure articles; LAWMA reports tricycle collection for areas trucks cannot reach | 6 of 8 households on untarred roads had waited more than 14 days, against 3 of 10 on good tarred roads; both households whose gate the truck cannot reach had waited more than 14 days | Consistent with the evidence; strongest survey pattern |
| H4 | Dumpsite bottleneck | Dumpsite access in 7 of 21 failure articles; failure reports are more frequent near disposal sites, an effect of reporting density (all articles show the same pattern) | Households with long gaps live in LGAs farther from a site (median 10.7 km against 7.3 km) | Partly consistent: dumpsite queues are reported, and distance aligns with longer gaps in the survey |
| H5 | Inequality | Not measured by news | Home type and gated estate show no clear pattern | Inconclusive; residents per operator range from 3,835 in Eti-Osa to 45,434 in Epe (40,430 in Ajeromi-Ifelodun), excluding Lagos Island |

## Limitations

- Both outcome measures are weak: news counts track reporting, and the survey is a 28-household convenience sample from 14 LGAs.
- Operator counts come from a January 2024 directory and do not record trucks, capacity or performance; Lagos Island is incomplete.
- Population figures are from 2006 and the two official counts disagree (C08).
- Distance is straight-line to three sites, two of them matched to unnamed map features (A09, A11); road distance and queue times are not measured.
- With 20 LGAs, rank correlations detect only large effects; the absence of a correlation does not rule an explanation out.